<script src="{{site.baseurl}}/assets/js/code-runner-analytics.js"></script>

## Popcorn Hack 1: Pick a Random SFI Part Record

`random.choice(parts)` hands back one whole dictionary from the list. That means the spec number, product name, and category of the chosen part always belong together.

In [ ]:
# CODE_RUNNER: Popcorn Hack 1 - Pick a Random SFI Part Record

import random

parts = [
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    },
    {
        "product_name": "Replacement Flywheels and Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    }
]

chosen_part = random.choice(parts)

print("Randomly chosen SFI part:")
print("  Spec number: ", chosen_part["spec_number"])
print("  Product name:", chosen_part["product_name"])
print("  Category:    ", chosen_part["category"])

## Popcorn Hack 2: Random SFI Record Number

The four records are numbered 1 to 4. `random.randint(1, record_count)` counts both the lowest and the highest number, so any record can come up. The runner proves this by drawing 100 times and keeping a tally for each record, then makes one final pick. Getting the same number more than once is expected, because each draw has nothing to do with the last one.

In [ ]:
# CODE_RUNNER: Popcorn Hack 2 - Random SFI Record Number

import random

record_count = 4

# Proof that any valid record ID can show up: draw 100 times and tally.
tally = {record: 0 for record in range(1, record_count + 1)}
for draw in range(100):
    tally[random.randint(1, record_count)] += 1

print("Draws per record over 100 runs:")
for record, count in tally.items():
    print("  Record", record, "->", count)
print("Every record came up at least once:", all(count > 0 for count in tally.values()))
print("Possible outputs:", list(range(1, record_count + 1)))

print()
record_id = random.randint(1, record_count)
print("SFI record picked for QA review:", record_id)

## Popcorn Hack 3: Random SFI API Test

For every test, the code picks a random action **and** a random part, then sends the action to its own backend-style reply. Three actions times three parts gives 9 possible combinations.

In [ ]:
# CODE_RUNNER: Popcorn Hack 3 - Random SFI API Test

import random

actions = [
    "PUT/update",
    "GET/search",
    "POST/create"
]

parts = [
    {"product_name": "Multiple Disc Clutch Assemblies", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "spec_number": "2.1"},
    {"product_name": "Replacement Flywheels", "spec_number": "1.1"}
]

for test in range(1, 6):
    action = random.choice(actions)
    part = random.choice(parts)

    if action == "PUT/update":
        response = "200 OK - spec " + part["spec_number"] + " updated"
    elif action == "GET/search":
        response = "200 OK - spec " + part["spec_number"] + " found"
    else:  # POST/create
        response = "201 Created - checked " + part["product_name"] + " before saving it"

    print("Test", test, "|", action, "|", part["product_name"], "->", response)

## Popcorn Hack 4 / Homework: SFI Backend QA Simulator

A reusable simulator that runs `test_count` random tests. In each test it picks a random action and part, then applies that action to the list of stored spec numbers:

| Action | Spec is stored | Spec is not stored |
| --- | --- | --- |
| `DELETE/remove` | 200: removed | 404: nothing to remove |
| `GET/search` | 200: record found | 404: not found |
| `PUT/update` | 200: updated | 404: nothing to update |
| `POST/create` | 409: duplicate spec, rejected | 201: created and stored |

Because the stored list changes during the run, a spec that gets created in an early test will be turned away as a duplicate if a later test tries to create it again.

In [ ]:
# CODE_RUNNER: Popcorn Hack 4 / Homework - SFI Backend QA Simulator

import random

actions = [
    "DELETE/remove",
    "GET/search",
    "PUT/update",
    "POST/create"
]

parts = [
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    },
    {
        "product_name": "Replacement Flywheels",
        "category": "Auto Racing",
        "spec_number": "1.1"
    }
]

existing_spec_numbers = ["2.1", "1.1"]
test_count = 6


def run_qa_test(part, action):
    """Run one action against the backend and return what happened."""
    spec = part["spec_number"]
    stored = spec in existing_spec_numbers

    if action == "DELETE/remove":
        if stored:
            existing_spec_numbers.remove(spec)
            return "200 OK: spec " + spec + " removed"
        return "404 Not Found: no spec " + spec + " to remove"

    if action == "GET/search":
        if stored:
            return "200 OK: " + part["product_name"] + " found"
        return "404 Not Found: spec " + spec + " is not in storage"

    if action == "PUT/update":
        if stored:
            return "200 OK: spec " + spec + " updated"
        return "404 Not Found: no spec " + spec + " to update"

    if action == "POST/create":
        if stored:
            return "409 Conflict: spec " + spec + " is already stored"
        existing_spec_numbers.append(spec)
        return "201 Created: spec " + spec + " saved"

    return "400 Bad Request: action " + action + " is not recognized"


def run_simulator(count):
    print("Specs at start:", existing_spec_numbers)
    print()
    for test_number in range(1, count + 1):
        action = random.choice(actions)
        part = random.choice(parts)
        result = run_qa_test(part, action)
        print("Test", test_number, "|", action, "|", part["spec_number"], part["product_name"])
        print("   ->", result)
    print()
    print("Specs at end:", existing_spec_numbers)
    print("No spec stored twice:", len(existing_spec_numbers) == len(set(existing_spec_numbers)))


run_simulator(test_count)